# RQ1 — Table 2: six matched baseline–plus pairs on the paper split

This reporting-only notebook recomputes the results of all six matched pairs on the fixed HEIG-Phish paper split (**1,657/553/553**). It does not train or tune a model.

LR and RF are additional standard baselines proposed by the group; they were not reported in the source paper. All values here are retrospective because this test split had previously been viewed.

In [1]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
from IPython.display import display

EXPECTED_MANIFEST_SHA256 = 'e31019521e64f6e105c17291a8e734b18a965448787ea3b459d8d3c88afec0a1'
EXPECTED_DATA_SHA256 = '8cc653bb389ea1a316f72c3893d0487ab09321552ae022ceaa89cd2499f690e8'

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def search_roots():
    roots = [Path.cwd(), Path('/kaggle/input'), Path('/kaggle/working'), Path(r'D:/Takedo/Documents/DAP/rq1_reporting_tables_20260926')]
    return [p for p in roots if p.exists()]

def locate_by_hash(name, expected_hash):
    matches = []
    for root in search_roots():
        for path in root.rglob(name):
            try:
                if sha256_file(path) == expected_hash:
                    matches.append(path)
            except OSError:
                pass
    unique = {str(p.resolve()): p for p in matches}
    assert unique, f'Cannot find {name} with SHA-256 {expected_hash}. Upload the complete reporting bundle.'
    return next(iter(unique.values()))

manifest_path = locate_by_hash('table2_full6_source_manifest.json', EXPECTED_MANIFEST_SHA256)
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest['source_data_sha256'] == EXPECTED_DATA_SHA256
assert manifest['paper_split'] == {
    'train': 1657, 'validation': 553, 'test': 553,
    'test_ids_sha256': '5ac043',
    'note': 'All six matched notebooks were independently audited as using the same paper-split test nodes. The abbreviated ID hash is retained as the cross-project audit label.'
}
assert manifest['model_seeds'] == [12, 13, 14, 15, 16]
source_dir = manifest_path.parent
print('Source directory:', source_dir)

Source directory: /kaggle/input/datasets/tranminhhuy15/adasdasdas/rq1_reporting_tables_final_3tables_20260926/sources/table2


In [2]:
for item in manifest['files']:
    path = source_dir / item['name']
    assert path.is_file(), path
    assert sha256_file(path) == item['sha256'], f'SHA-256 mismatch: {path}'
print(f"Integrity check passed for {len(manifest['files'])} matched result files.")

Integrity check passed for 6 matched result files.


## Recompute all metrics from the five runs

Every cell reports mean ± sample SD over seeds 12–16. The plus-minus variation here measures model-run randomness on one fixed split; it does not measure uncertainty over different data splits.

In [3]:
model_order = ['LR', 'RF', 'LightGBM', 'GAT', 'GraphSAGE', 'Meta-IFD']
metric_columns = {
    'Precision': 'test_precision',
    'Recall': 'test_recall',
    'Binary-F1': 'test_binary_f1',
    'Micro-F1': 'test_micro_f1',
    'Macro-F1': 'test_macro_f1',
    'PR-AUC': 'test_pr_auc',
    'FPR': 'test_fpr',
}

frames = []
for item in manifest['files']:
    df = pd.read_csv(source_dir / item['name'])
    assert len(df) == 10
    assert set(df['split_key']) == {'A_paper_split'}
    assert set(df['variant']) == {'base', 'plus'}
    assert set(df['seed'].astype(int)) == {12, 13, 14, 15, 16}
    assert ((df[['test_tn', 'test_fp', 'test_fn', 'test_tp']].sum(axis=1)) == 553).all()
    frames.append(df)
runs = pd.concat(frames, ignore_index=True)
assert len(runs) == 60
assert runs.groupby(['model', 'variant']).size().eq(5).all()

pair_alias = {
    'LR': ('LR', 'LR+'),
    'RF': ('RF', 'RF+'),
    'LightGBM': ('LightGBM', 'LightGBM+'),
    'GAT': ('GAT-Norm', 'GAT+'),
    'GraphSAGE': ('GraphSAGE-Norm', 'GraphSAGE+'),
    'Meta-IFD': ('Meta-IFD', 'Meta-IFD+'),
}
rows = []
for pair in model_order:
    base_name, plus_name = pair_alias[pair]
    base_macro = None
    for variant, display_name, role in [('base', base_name, 'Baseline'), ('plus', plus_name, 'Plus')]:
        part = runs[(runs['model'] == display_name) & (runs['variant'] == variant)]
        assert len(part) == 5, (pair, variant, len(part))
        row = {'Pair': pair, 'Model': display_name, 'Role': role}
        for label, column in metric_columns.items():
            values = part[column].to_numpy(dtype=float) * 100
            row[f'{label} mean'] = values.mean()
            row[f'{label} SD'] = values.std(ddof=1)
        if variant == 'base':
            base_macro = row['Macro-F1 mean']
            row['Delta Macro-F1 vs baseline (pp)'] = 0.0
        else:
            row['Delta Macro-F1 vs baseline (pp)'] = row['Macro-F1 mean'] - base_macro
        rows.append(row)
table_numeric = pd.DataFrame(rows)
display(table_numeric)

,Pair,Model,Role,Precision mean,Precision SD,Recall mean,Recall SD,Binary-F1 mean,Binary-F1 SD,Micro-F1 mean,Micro-F1 SD,Macro-F1 mean,Macro-F1 SD,PR-AUC mean,PR-AUC SD,FPR mean,FPR SD,Delta Macro-F1 vs baseline (pp)
0,LR,LR,Baseline,84.397163,0.000000,98.755187,0.000000,91.013384,0.000000,91.500904,0.000000,91.475817,0.000000,94.471533,0.000000,14.102564,1.986027e-15,0.000000
1,LR,LR+,Plus,96.721311,0.000000,97.925311,0.000000,97.319588,0.000000,97.649186,0.000000,97.613095,0.000000,99.308299,0.000000,2.564103,0.000000e+00,6.137278
2,RF,RF,Baseline,94.401208,0.377607,97.925311,0.000000,96.130665,0.195786,96.564195,0.180832,96.520508,0.181706,98.801847,0.072507,4.487179,3.205128e-01,0.000000
3,RF,RF+,Plus,99.579832,0.000000,98.340249,0.000000,98.956159,0.000000,99.095841,0.000000,99.079355,0.000000,99.833951,0.019152,0.320513,0.000000e+00,2.558847
4,LightGBM,LightGBM,Baseline,93.650794,0.000000,97.925311,0.000000,95.740365,0.000000,96.202532,0.000000,96.157295,0.000000,97.876302,0.000000,5.128205,0.000000e+00,0.000000
5,LightGBM,LightGBM+,Plus,99.579832,0.000000,98.340249,0.000000,98.956159,0.000000,99.095841,0.000000,99.079355,0.000000,99.868345,0.000000,0.320513,0.000000e+00,2.922060
6,GAT,GAT-Norm,Baseline,81.514693,3.555203,92.116183,2.992159,86.465143,2.871321,87.414105,2.759093,87.349028,2.758909,89.262010,3.357556,16.217949,3.515428e+00,0.000000
7,GAT,GAT+,Plus,97.413439,0.535398,96.846473,0.629284,97.128356,0.498922,97.504521,0.431730,97.460940,0.439696,99.449587,0.103786,1.987179,4.178976e-01,10.111911
8,GraphSAGE,GraphSAGE-Norm,Baseline,90.680543,0.971346,96.016598,1.082025,93.268709,0.790585,93.960217,0.707328,93.895636,0.715007,96.791861,0.247863,7.628205,8.600261e-01,0.000000
9,GraphSAGE,GraphSAGE+,Plus,99.568583,0.002431,95.767635,0.541013,97.630510,0.282559,97.974684,0.235776,97.931025,0.242347,99.710498,0.015657,0.320513,0.000000e+00,4.035389


In [4]:
def mean_sd(row, metric):
    return f"{row[f'{metric} mean']:.2f} ± {row[f'{metric} SD']:.2f}"

table_display = table_numeric[['Model', 'Role']].copy()
for metric in ['Precision', 'Recall', 'Binary-F1', 'Micro-F1', 'Macro-F1', 'PR-AUC', 'FPR']:
    table_display[metric] = table_numeric.apply(lambda r: mean_sd(r, metric), axis=1)
table_display['Δ Macro (pp)'] = table_numeric.apply(
    lambda r: '—' if r['Role'] == 'Baseline' else f"{r['Delta Macro-F1 vs baseline (pp)']:+.2f}", axis=1
)
display(table_display.style.hide(axis='index').set_caption(
    'Table 2. Six matched baseline–plus comparisons on the fixed HEIG-Phish paper split (%)'
))

Model,Role,Precision,Recall,Binary-F1,Micro-F1,Macro-F1,PR-AUC,FPR,Δ Macro (pp)
LR,Baseline,84.40 ± 0.00,98.76 ± 0.00,91.01 ± 0.00,91.50 ± 0.00,91.48 ± 0.00,94.47 ± 0.00,14.10 ± 0.00,—
LR+,Plus,96.72 ± 0.00,97.93 ± 0.00,97.32 ± 0.00,97.65 ± 0.00,97.61 ± 0.00,99.31 ± 0.00,2.56 ± 0.00,+6.14
RF,Baseline,94.40 ± 0.38,97.93 ± 0.00,96.13 ± 0.20,96.56 ± 0.18,96.52 ± 0.18,98.80 ± 0.07,4.49 ± 0.32,—
RF+,Plus,99.58 ± 0.00,98.34 ± 0.00,98.96 ± 0.00,99.10 ± 0.00,99.08 ± 0.00,99.83 ± 0.02,0.32 ± 0.00,+2.56
LightGBM,Baseline,93.65 ± 0.00,97.93 ± 0.00,95.74 ± 0.00,96.20 ± 0.00,96.16 ± 0.00,97.88 ± 0.00,5.13 ± 0.00,—
LightGBM+,Plus,99.58 ± 0.00,98.34 ± 0.00,98.96 ± 0.00,99.10 ± 0.00,99.08 ± 0.00,99.87 ± 0.00,0.32 ± 0.00,+2.92
GAT-Norm,Baseline,81.51 ± 3.56,92.12 ± 2.99,86.47 ± 2.87,87.41 ± 2.76,87.35 ± 2.76,89.26 ± 3.36,16.22 ± 3.52,—
GAT+,Plus,97.41 ± 0.54,96.85 ± 0.63,97.13 ± 0.50,97.50 ± 0.43,97.46 ± 0.44,99.45 ± 0.10,1.99 ± 0.42,+10.11
GraphSAGE-Norm,Baseline,90.68 ± 0.97,96.02 ± 1.08,93.27 ± 0.79,93.96 ± 0.71,93.90 ± 0.72,96.79 ± 0.25,7.63 ± 0.86,—
GraphSAGE+,Plus,99.57 ± 0.00,95.77 ± 0.54,97.63 ± 0.28,97.97 ± 0.24,97.93 ± 0.24,99.71 ± 0.02,0.32 ± 0.00,+4.04


### Interpretation guardrail

This table provides a descriptive comparison on the fixed paper split. It must not be used as the confirmatory significance test because the split had already been viewed. Table 3 provides the preregistered 20-resplit evidence.

In [5]:
output_dir = Path('/kaggle/working/rq1_table_outputs/table2_full6') if Path('/kaggle/working').exists() else Path.cwd() / 'rq1_table_outputs' / 'table2_full6'
output_dir.mkdir(parents=True, exist_ok=True)
table_numeric.to_csv(output_dir / 'table2_six_pairs_paper_split_numeric.csv', index=False)
table_display.to_csv(output_dir / 'table2_six_pairs_paper_split_display.csv', index=False)

def dataframe_to_markdown(frame):
    clean = frame.astype(str).map(lambda x: x.replace('|', chr(92) + '|'))
    lines = ['| ' + ' | '.join(clean.columns) + ' |', '|' + '|'.join(['---'] * len(clean.columns)) + '|']
    lines.extend('| ' + ' | '.join(row) + ' |' for row in clean.to_numpy().tolist())
    return '\n'.join(lines) + '\n'

(output_dir / 'table2_six_pairs_paper_split.md').write_text(dataframe_to_markdown(table_display), encoding='utf-8')
(output_dir / 'table2_six_pairs_paper_split.tex').write_text(
    table_display.to_latex(index=False, escape=True, caption='Six matched baseline--plus comparisons on the fixed HEIG-Phish paper split (percent).', label='tab:paper-split-six-pairs'),
    encoding='utf-8'
)
metadata = {
    'table': 2,
    'scientific_role': manifest['scientific_role'],
    'manifest_sha256': EXPECTED_MANIFEST_SHA256,
    'source_data_sha256': EXPECTED_DATA_SHA256,
    'paper_split': manifest['paper_split'],
    'model_seeds': manifest['model_seeds'],
    'models': table_display['Model'].tolist(),
}
(output_dir / 'table2_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Saved Table 2 artifacts to', output_dir)

Saved Table 2 artifacts to /kaggle/working/rq1_table_outputs/table2_full6
